In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets
from torchvision.transforms import v2
import torchvision.transforms.functional as FF
import torch.nn.functional as F
import random

In [2]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        self.Encoder = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.Conv2d(64, 64, 4, 2, 1),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.Conv2d(256, 512, 4, 2, 1),
        )
        
        self.Decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ConvTranspose2d(64, 3, 4, 2, 1)
        )

        self.Discriminator = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.Conv2d(128, 256, 4, 2, 1),
            nn.Conv2d(256, 512, 4, 2, 1),
            nn.Conv2d(512, 1, 4, 2, 1)
        )

        self.EncoderConv = nn.Conv2d(512, 40000, 4 , 1)
        self.DecoderUconv = nn.ConvTranspose2d(4000, 512, 4, 1)
    
    def forward(self, img, drop):
        latent = torch.flatten(self.EncoderConv(self.Encoder(img)), dim=1)
        output = self.Decoder(self.DecoderUconv(latent.view(-1, 4000, 1, 1)))
        rf = self.Discriminator(drop)

        return output, rf